### Ingestão de dados na Camada Bronze

Ingestão dos arquivos CSV históricos das tabelas atribuídas disponíveis no container `raw/batch-data` para a camada Bronze.

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col,lit,current_timestamp,year,month,max as spark_max
from pyspark.sql.types import StructType,StructField,StringType,LongType,TimestampType
from pyspark.errors import AnalysisException

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")


# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data",
    "food_avaliacoes_produto_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_avaliacoes_produto",
    "food_estoque_lojas_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_estoque_lojas",
    "food_avaliacoes_produto_log": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/metadata/metadata_ingestion_log/food_avaliacoes_produto",
    "food_estoques_lojas_log": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/metadata/metadata_ingestion_log/food_estoque_lojas"
}

# DICIONÁRIO DE INFORMAÇÕES DAS TABELAS
tables_config = {
    "food_avaliacoes_produto": {
        "file_pattern": "food_avaliacoes_produto",
        "date_column": "dt_avaliacao",
        "bronze_path": paths["food_avaliacoes_produto_bronze"],
        "log_path": paths["food_avaliacoes_produto_log"]
    },

    "food_estoque_lojas": {
        "file_pattern": "food_estoque_lojas",
        "date_column": "dt_snapshot",
        "bronze_path": paths["food_estoque_lojas_bronze"],
        "log_path": paths["food_estoques_lojas_log"]
    }
}

# VERIFICA SE EXISTE REGISTRO DE ARQUIVO PROCESSADO
def log_exists(path):
    try:
        (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(path)
            .limit(1)
            .count()
        )
        return True
    except Exception:
        return False
    

In [0]:
# INGESTÃO BRONZE COM CONTROLE DE LOG
def ingest_bronze(table_name: str):
    # 1. Captura metadados de todos os arquivos da RAW que contém o file_pattern em seu nome
    raw_file_info = (
        spark.read
        .format("binaryFile")
        .options(**adls_options)
        .load(paths["raw_path"])
        .filter(col("path").contains(tables_config[table_name]["file_pattern"]))
        .select(col("path").alias("file_path"),col("modificationTime").alias("source_modification_time"))
    )

    # 2. Verifica a existência de algum log de controle
    if log_exists(tables_config[table_name]["log_path"]):
        df_log = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(tables_config[table_name]["log_path"])
        )

        # 3. Identifica arquivos/versões ainda não processados a partir do log de controle
        df_new_raw_files = (
            raw_file_info.alias("raw")
            .join(
                df_log.alias("log"),
                (col("raw.file_path") == col("log.processed_file")) &
                (col("raw.source_modification_time") == col("log.source_modification_time")),
                "left_anti"
            )
        )

    else:
        print(f"[PRIMEIRA INGESTÃO]. Não existe log de ingestão para {table_name}.Todos os dados serão processados")
        df_new_raw_files = raw_file_info

    # 4. Verifica existência de arquivos para serem processados após comparação dos metadados de RAW com logs
    if df_new_raw_files.limit(1).count() == 0:
        print(f"[SEM ALTERAÇÃO].{table_name} não possui novos arquivos para ingestão.")
        return None, None
    
    # 5. Obtém os caminhos dos arquivos a serem processados
    new_files_to_process = [
        row["file_path"]
        for row in df_new_raw_files.select("file_path").collect()
    ]

    # 6. Le Arquivos de RAW
    df = (
        spark.read
        .options(**adls_options)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(new_files_to_process)
    )
    raw_count = df.count()

    # 7. Adiciona colunas de auditoria e colunas para particionamento
    bronze_ingested_at = current_timestamp()
    df = (
        df
        .withColumn("bronze_ingested_at", bronze_ingested_at)
        .withColumn("bronze_source_file", col("_metadata.file_path"))
        .withColumn("ano", year(col(tables_config[table_name]["date_column"])))
        .withColumn("mes", month(col(tables_config[table_name]["date_column"])))
    )

    # 8. Obtém a quantidade de linhas processadas por arquivo
    df_file_processed_raw_counts = (
        df
        .groupBy("bronze_source_file")
        .count()
        .withColumnRenamed("count", "total_rows")
    )

    # 9. Obtém a data de modificação dos arquivos processados
    df_new_raw_file_metadata = (
        df_new_raw_files
        .select(
            col("file_path").alias("processed_file"),
            col("source_modification_time")
        )
    )

    # 10. Cria o DataFrame do log de controle
    df_new_log = (
        df_file_processed_raw_counts.alias("count")
        .join(
            df_new_raw_file_metadata.alias("metadata"),
            col("count.bronze_source_file") == col("metadata.processed_file"),
            "inner"
        )
        .select(
            lit(table_name).alias("table_name"),
            col("metadata.processed_file"),
            col("count.total_rows"),
            lit("SUCCESS").alias("status"),
            bronze_ingested_at.alias("bronze_ingested_at"),
            col("metadata.source_modification_time")
        )
    )

    # 11. Salva Delta append
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("ano", "mes")
        .options(**adls_options)
        .save(tables_config[table_name]["bronze_path"])
    )

    # 12. Salva log da tabela
    (
        df_new_log.write
        .format("delta")
        .mode("append")
        .options(**adls_options)
        .save(tables_config[table_name]["log_path"])
    )

    print(f"[INGESTÃO CONCLUÍDA] {table_name} | {raw_count} registros")
    return df, df_new_log
    

In [0]:
# EXECUÇÃO DAS TABELAS BRONZE
df_bronze_avaliacoes, df_log_avaliacoes = ingest_bronze("food_avaliacoes_produto")
df_bronze_estoque, df_log_estoque = ingest_bronze("food_estoque_lojas")

# VISUALIZAÇÃO DOS DATAFRAMES BRONZE
display(df_bronze_avaliacoes)
display(df_bronze_estoque)

# VISUALIZAÇÃO DOS LOGS
display(df_log_avaliacoes)
display(df_log_estoque)